In [ ]:
%run ./NB_Config_SELLING_SYSTEM

from datetime import datetime
import uuid
from pyspark.sql import functions as F

PIPELINE_RUN_ID = str(uuid.uuid4())
start_time = datetime.utcnow()
results = []

for table_name in FINAL_TABLES:
    silver_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{table_name}'
    gold_path = f'{GOLD_LH_ABFSS}/{GOLD_SCHEMA}/{table_name}'
    df = spark.read.format('delta').load(silver_path)
    data_columns = [c for c in FINAL_COLUMNS[table_name] if c in df.columns]
    hash_expr = F.sha2(F.concat_ws('||', *[F.coalesce(F.col(c).cast('string'), F.lit('')) for c in data_columns]), 256)
    (df.withColumn('_ROW_HASH', hash_expr)
       .withColumn('_PIPELINE_NAME', F.lit(PIPELINE_NAME))
       .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
       .withColumn('_PUBLISHED_AT', F.current_timestamp())
       .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(gold_path))
    rows = df.count()
    results.append({'table': table_name, 'rows': rows})
    print(f'{table_name}: published {rows:,} rows')

duration = (datetime.utcnow() - start_time).total_seconds()
print('\nSELLING SYSTEM GOLD SUMMARY')
print(f'Tables: {len(results)} | Duration: {duration:.1f}s')